# Bloom Filters

## Learning Objectives

1. **Describe** the Bloom filter data structure and its one-sided error guarantee
2. **Derive** the optimal number of hash functions given $n$ and $m$
3. **Implement** a Bloom filter using bit arrays and multiple hash functions
4. **Compute** the false positive probability analytically

## Motivation: Set Membership at Scale

**Problem:** given a set $S$ of $n$ items, answer membership queries $x \in S?$ with limited memory.

**Exact hash set:** $O(n)$ memory, zero errors.

**Bloom filter:** $O(m)$ bits where $m \ll n \cdot \text{pointer size}$, with a small false positive rate.

**One-sided guarantee:**
- $x \in S \Rightarrow$ always returns TRUE (no false negatives)
- $x \notin S \Rightarrow$ returns TRUE with probability $\epsilon$ (false positive rate)

Applications: web crawlers (avoid re-crawling), database join pre-filtering, spam detection, CDN cache admission.

## Construction

**Structure:** a bit array $B[0 \ldots m-1]$, initially all zeros.

**$k$ independent hash functions:** $h_1, h_2, \ldots, h_k : U \to \{0, \ldots, m-1\}$

**Insert element $x$:**
For each $i \in \{1,\ldots,k\}$: set $B[h_i(x)] = 1$

**Query element $y$:**
Return TRUE iff $B[h_i(y)] = 1$ for all $i \in \{1,\ldots,k\}$

**False positive:** $y \notin S$ but all $k$ positions happen to be 1 due to other insertions.

In [1]:
import math, hashlib

class BloomFilter:
    def __init__(self, n, fp_rate=0.01):
        # Optimal m and k for given n elements and target false positive rate
        self.m = int(-n * math.log(fp_rate) / (math.log(2)**2))
        self.k = int(self.m / n * math.log(2))
        self.bits = [False]*self.m
        self.n_inserted = 0
        print(f"BloomFilter: n={n}, fp_rate={fp_rate}")
        print(f"  m={self.m} bits ({self.m/8/1024:.1f} KB), k={self.k} hash functions")

    def _hashes(self, item):
        item_bytes = str(item).encode()
        for i in range(self.k):
            h = int(hashlib.md5(item_bytes + i.to_bytes(4,'big')).hexdigest(), 16)
            yield h % self.m

    def insert(self, item):
        for pos in self._hashes(item):
            self.bits[pos] = True
        self.n_inserted += 1

    def query(self, item):
        return all(self.bits[pos] for pos in self._hashes(item))

    def fp_probability(self):
        """Analytical false positive probability given current fill."""
        load = self.n_inserted / self.m
        return (1 - math.exp(-self.k * load))**self.k

bf = BloomFilter(n=1000, fp_rate=0.01)
S = set(range(1000))
for x in S:
    bf.insert(x)

# Test false positives on items NOT in S
queries_not_in_S = range(10000, 11000)
fp_count = sum(1 for x in queries_not_in_S if bf.query(x))
observed_fp = fp_count / len(list(queries_not_in_S))

print(f"""
False positive rate:""")
print(f"  Analytical: {bf.fp_probability():.4f}")
print(f"  Observed:   {observed_fp:.4f}")
print(f"""
Membership check for x=500 (in S): {bf.query(500)}""")
print(f"Membership check for x=9999 (not in S): {bf.query(9999)}")

BloomFilter: n=1000, fp_rate=0.01
  m=9585 bits (1.2 KB), k=6 hash functions

False positive rate:
  Analytical: 0.0101
  Observed:   0.0120

Membership check for x=500 (in S): True
Membership check for x=9999 (not in S): False


In [2]:
# FP rate vs bits per element (m/n)
print("bits/element (m/n)  |  k_opt  |  FP rate")
print("-"*45)
for bits_per_elem in [4, 6, 8, 10, 12, 14, 16]:
    ratio = bits_per_elem
    k_opt = round(ratio * math.log(2))
    fp = (1 - math.exp(-k_opt/ratio))**k_opt
    bar = '#'*int(fp*200)
    print(f"  {bits_per_elem:>4}            |  {k_opt:>5}  |  {fp:.4f}  {bar}")

# Memory comparison: exact set vs Bloom filter
n = 1_000_000
print(f"""
For n = {n:,} items:""")
print(f"  Python set (64-bit pointers):  {n*64/8/1e6:.0f} MB")
bloom_bits = int(-n*math.log(0.01)/(math.log(2)**2))
print(f"  Bloom filter (FP=1%):          {bloom_bits/8/1e6:.1f} MB")
print(f"  Bloom filter (FP=0.1%):        {int(-n*math.log(0.001)/(math.log(2)**2))/8/1e6:.1f} MB")

bits/element (m/n)  |  k_opt  |  FP rate
---------------------------------------------
     4            |      3  |  0.1469  #############################
     6            |      4  |  0.0561  ###########
     8            |      6  |  0.0216  ####
    10            |      7  |  0.0082  #
    12            |      8  |  0.0031  
    14            |     10  |  0.0012  
    16            |     11  |  0.0005  

For n = 1,000,000 items:
  Python set (64-bit pointers):  8 MB
  Bloom filter (FP=1%):          1.2 MB
  Bloom filter (FP=0.1%):        1.8 MB


## Optimal Parameters

Given $n$ (expected insertions) and target false positive rate $\epsilon$:

$$m^* = -\frac{n \ln \epsilon}{(\ln 2)^2}$$

$$k^* = \frac{m}{n} \ln 2$$

The resulting false positive rate is exactly $\epsilon$.

**Rule of thumb:** 10 bits per element → 1% false positive rate with 7 hash functions.

## Limitations
- No deletions (setting bits to 0 can break other elements' membership)
- Counting Bloom filter: use counters instead of bits — allows deletions but 4× memory